# Using Lightgbm  Only

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

In [ ]:
print("1. Loading Data...")
# Replace paths if needed!
train = pd.read_csv('/content/drive/MyDrive/Playground-series/s6e3/train.csv')
test = pd.read_csv('/content/drive/MyDrive/Playground-series/s6e3/test.csv')

TARGET = 'Churn'
ID_COL = 'id'

In [ ]:
y = train[TARGET].map({'Yes': 1, 'No': 0})
X = train.drop(columns=[TARGET, ID_COL])
X_test = test.drop(columns=[ID_COL])

In [ ]:
print("2. Applying Feature Engineering...")
# --- THE SECRET SAUCE: FEATURE ENGINEERING ---
def apply_feature_engineering(df):
    df = df.copy()

    # TotalCharges has empty strings " " for new customers. Convert to numeric and fill with 0.
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

    # 1. Financial Features
    # Compare what they should have paid vs what they actually paid
    df['Calculated_Total'] = df['tenure'] * df['MonthlyCharges']
    df['Total_Difference'] = df['TotalCharges'] - df['Calculated_Total']

    # 2. Service Counts (More services = less likely to churn usually)
    security_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    # Count how many of these features they have
    df['Security_Count'] = sum((df[col] == 'Yes').astype(int) for col in security_cols)

    streaming_cols = ['StreamingTV', 'StreamingMovies']
    df['Streaming_Count'] = sum((df[col] == 'Yes').astype(int) for col in streaming_cols)

    # Total number of services they subscribe to
    df['Total_Services'] = df['Security_Count'] + df['Streaming_Count'] + (df['PhoneService'] == 'Yes').astype(int)

    # 3. Ratios
    # How much are they paying per service?
    df['Cost_Per_Service'] = df['MonthlyCharges'] / (df['Total_Services'] + 1)

    return df

In [ ]:
# Apply to both Train and Test
X = apply_feature_engineering(X)
X_test = apply_feature_engineering(X_test)

In [ ]:
print("3. Preprocessing Categoricals...")
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

X_lgb_xgb = X.copy()
X_test_lgb_xgb = X_test.copy()
X_cat = X.copy()
X_test_cat = X_test.copy()

if len(cat_cols) > 0:
    # A. CatBoost Data (Raw Strings)
    X_cat[cat_cols] = X_cat[cat_cols].astype(str).fillna('Missing')
    X_test_cat[cat_cols] = X_test_cat[cat_cols].astype(str).fillna('Missing')

    # B. LGBM/XGB Data (Ordinal Encoded)
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_lgb_xgb[cat_cols] = X_lgb_xgb[cat_cols].astype(str).fillna('Missing')
    X_test_lgb_xgb[cat_cols] = X_test_lgb_xgb[cat_cols].astype(str).fillna('Missing')

    X_lgb_xgb[cat_cols] = encoder.fit_transform(X_lgb_xgb[cat_cols])
    X_test_lgb_xgb[cat_cols] = encoder.transform(X_test_lgb_xgb[cat_cols])

    # Clean column names for LightGBM
X_lgb_xgb.columns = ["".join (c if c.isalnum() else "_" for c in str(x)) for x in X_lgb_xgb.columns]
X_test_lgb_xgb.columns = ["".join (c if c.isalnum() else "_" for c in str(x)) for x in X_test_lgb_xgb.columns]

In [ ]:
print("4. Starting Model Training (5-Fold CV) - LightGBM ONLY...")
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Only setting up arrays for LightGBM
lgb_oof = np.zeros(len(X))
lgb_test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n========== FOLD {fold+1} ==========")
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # We only need the LGBM/XGB (Ordinal Encoded) data split
    X_train_lx, X_val_lx = X_lgb_xgb.iloc[train_idx], X_lgb_xgb.iloc[val_idx]

    # --- MODEL: LIGHTGBM ---
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42+fold,
        verbose=-1
    )

    model_lgb.fit(
        X_train_lx, y_train,
        eval_set=[(X_val_lx, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=75, verbose=False)]
    )

    # Save Out-Of-Fold predictions
    fold_preds = model_lgb.predict_proba(X_val_lx)[:, 1]
    lgb_oof[val_idx] = fold_preds

    # Add test predictions (divided by N_FOLDS to average them)
    lgb_test_preds += model_lgb.predict_proba(X_test_lgb_xgb)[:, 1] / N_FOLDS

    print(f"LGBM Fold {fold+1} AUC: {roc_auc_score(y_val, fold_preds):.5f}")

In [ ]:
print("\n======================================")
final_auc = roc_auc_score(y, lgb_oof)
print(f"🏆 OVERALL LGBM OOF AUC: {final_auc:.5f}")

# --- 5. CREATE SUBMISSION FILE ---
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: lgb_test_preds
})

submission.to_csv('submission_lgbm_only.csv', index=False)
print("submission_lgbm_only.csv created successfully! Ready to upload to Kaggle.")